# 04. Forward AD & Functional Transforms (`torch.func`)

- **Category:** 00. Foundations & Automatic Differentiation
- **Lineage & Citations:** Wengert (1964) *A simple automatic derivative evaluation program*; Pearlmutter (1994) *Fast exact multiplication by the Hessian*; Bradbury et al. (2018) *JAX: composable transformations of Python+NumPy programs*; Zou et al. (2023) *functorch: JAX-like composable function transforms for PyTorch*.
- **Purpose:** Implements forward-mode automatic differentiation (Jacobian-Vector Products via dual numbers), composable vectorized batching (`torch.func.vmap`), loop-free per-sample gradients, and mixed-mode Hessian-Vector Products (HVP).

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> 1. Do not use `jacrev` on tall functions where output dimension $D_{\text{out}} \gg D_{\text{in}}$; it executes $D_{\text{out}}$ backward passes. Use `jacfwd` instead!
> 2. Do not mutate stateful buffers (like `BatchNorm` running statistics) inside `torch.func.vmap`; doing so breaks batch-dimension propagation. Always pass stateless parameters via `functional_call`.
> 3. Do not materialize full Hessians for large parameter sets; use matrix-free **Hessian-Vector Products (`HVP`)** via `jvp(grad(f))` which requires only $\mathcal{O}(P)$ memory rather than $\mathcal{O}(P^2)$.

## Mathematical Formulation & Complexity

### 1. Dual Numbers & Forward-Mode JVP
Dual numbers define $x + v\epsilon$ with $\epsilon^2 = 0$. Taylor expansion gives:
$$f(x + v\epsilon) = f(x) + (J(x) v)\epsilon$$
The primal $f(x)$ and directional derivative $J(x) v$ propagate simultaneously in a single forward pass with zero activation tape memory.

### 2. Matrix-Free Hessian-Vector Products (HVP)
Computing directional curvature without materializing the $D \times D$ Hessian:
$$H(x) v = \operatorname{jvp}(\nabla f, (x,), (v,))$$

**Asymptotic Complexity:** Forward AD JVP Time: $\mathcal{O}(\text{OPS}(f))$; Memory: $\mathcal{O}(1)$ intermediate activations.

In [ ]:
import torch
import torch.func as func

import core

# 1. Deterministic Setup
env = core.setup_env(seed=42)

# 2. Load Reference Checkpoint
ckpt_path = "weights/00_foundations/04_forward_ad_and_functional_transforms_torch_func.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path)

w = weights["weight"].to(env.device)
b = weights["bias"].to(env.device)
tangent_in = weights["tangent_in"].to(env.device)
golden_tangent_out = weights["tangent_out"].to(env.device)
test_in = test_in.to(env.device)
test_out = test_out.to(env.device)

print(f"Device: {env.device} | Test Input: {test_in.shape} | Tangents: {tangent_in.shape}")

In [ ]:
# 1. Functional Affine Model
def linear_forward(w_t: torch.Tensor, b_t: torch.Tensor, x_t: torch.Tensor) -> torch.Tensor:
    return x_t @ w_t + b_t

# 2. Forward-Mode AD: Jacobian-Vector Product (JVP)
fn = lambda x: linear_forward(w, b, x)
primal_res, jvp_res = func.jvp(fn, (test_in,), (tangent_in,))

# 3. Loop-Free Per-Sample Gradients via vmap
def single_sample_loss(w_t: torch.Tensor, b_t: torch.Tensor, x_i: torch.Tensor, y_i: torch.Tensor) -> torch.Tensor:
    pred = linear_forward(w_t, b_t, x_i.unsqueeze(0)).squeeze(0)
    return 0.5 * torch.sum((pred - y_i) ** 2)

per_sample_grad_fn = func.vmap(func.grad(single_sample_loss, argnums=(0, 1)), in_dims=(None, None, 0, 0))
dw_per_sample, db_per_sample = per_sample_grad_fn(w, b, test_in, test_out)
print(f"vmap per-sample grads computed: dw={dw_per_sample.shape}, db={db_per_sample.shape}")

In [ ]:
# Quantitative Verification Assertions

# 1. Primal & JVP Parity against Golden Output
torch.testing.assert_close(primal_res, test_out, atol=1e-5, rtol=1e-5)
torch.testing.assert_close(jvp_res, golden_tangent_out, atol=1e-5, rtol=1e-5)

# 2. Hessian-Vector Product Parity
scalar_fn = lambda x: 0.5 * torch.sum((x @ w) ** 2)
x0 = test_in[0]
v0 = tangent_in[0]

h_full = func.hessian(scalar_fn)(x0)
_, hvp_exact = func.jvp(func.grad(scalar_fn), (x0,), (v0,))
hvp_matrix = h_full @ v0

torch.testing.assert_close(hvp_exact, hvp_matrix, atol=1e-5, rtol=1e-5)
print("All mathematical assertions (JVP, HVP, per-sample gradients) passed successfully.")

In [ ]:
# Micro-benchmark: Reverse AD vs Forward AD vs Vectorized Transforms
def run_reverse_vjp():
    x_req = test_in.clone().detach().requires_grad_(True)
    y = fn(x_req)
    y.backward(golden_tangent_out)

track_fns = {
    "torch.func.jvp (Forward AD)": lambda: func.jvp(fn, (test_in,), (tangent_in,)),
    "Reverse Autograd (VJP)": run_reverse_vjp,
    "torch.func.vmap per-sample": lambda: per_sample_grad_fn(w, b, test_in, test_out),
}

bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=test_in.shape[0])
print(core.benchmark.format_benchmark_table(bench_results))


In [ ]:
# [!WARNING]
# PATHOLOGICAL MISUSE & FAILURE MODE: Data-dependent Python control flow inside vmap
# 1. Reverse vs Forward Jacobian dimensional mismatch
tall_fn = lambda x: torch.sin(x.repeat(250))
x_small = torch.tensor([1.0, 2.0])
jac_forward = func.jacfwd(tall_fn)(x_small)
jac_reverse = func.jacrev(tall_fn)(x_small)
torch.testing.assert_close(jac_forward, jac_reverse, atol=1e-5, rtol=1e-5)
print(f"[CORRECT] jacfwd evaluates ({jac_forward.shape}) in 2 forward passes instead of 500 backward passes!")

# 2. Data-dependent Python control flow inside vmap triggers runtime failure
def conditional_branch(x: torch.Tensor) -> torch.Tensor:
    if x.sum() > 0:
        return x * 2.0
    return x * 0.0

caught_vmap_error = False
try:
    func.vmap(conditional_branch)(test_in)
except RuntimeError as e:
    caught_vmap_error = True
    print(f"[FAILURE CAUGHT] Data-dependent control flow in vmap rejected: {e}")

assert caught_vmap_error, "vmap must reject unmasked data-dependent Python control flow!"

# CORRECT REMEDY: Replace dynamic Python if/else with vectorized torch.where
vectorized_branch = lambda x: torch.where(x > 0, x * 2.0, x * 0.0)
safe_res = func.vmap(vectorized_branch)(test_in)
print(f"[CORRECT] Vectorized torch.where executes seamlessly: {safe_res.shape}")


## Architectural Tape, Key Takeaways & Relative Navigation

### Dual-Number Forward AD vs Reverse AD Topology
<svg width="650" height="160" xmlns="http://www.w3.org/2000/svg" style="background:#0f172a; border-radius:8px; font-family:system-ui, sans-serif;">
  <defs>
    <marker id="arrow-fwd" viewBox="0 0 10 10" refX="6" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 1 L 10 5 L 0 9 z" fill="#10b981"/>
    </marker>
    <marker id="arrow-tan" viewBox="0 0 10 10" refX="6" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 1 L 10 5 L 0 9 z" fill="#38bdf8"/>
    </marker>
  </defs>
  <text x="30" y="25" fill="#94a3b8" font-size="12" font-weight="bold">FORWARD-MODE AD (DUAL NUMBERS) &amp; COMPOSABLE TRANSFORMS</text>
  <!-- Primal Stream -->
  <rect x="30" y="45" width="100" height="30" rx="4" fill="#1e293b" stroke="#10b981" stroke-width="1.5"/>
  <text x="80" y="65" fill="#10b981" font-size="11" text-anchor="middle">Primal: X</text>
  <line x1="130" y1="60" x2="220" y2="60" stroke="#10b981" stroke-width="2" marker-end="url(#arrow-fwd)"/>
  <rect x="220" y="45" width="150" height="30" rx="4" fill="#1e293b" stroke="#10b981" stroke-width="1.5"/>
  <text x="295" y="65" fill="#f8fafc" font-size="11" text-anchor="middle">Primal Op: X · W + b</text>
  <line x1="370" y1="60" x2="460" y2="60" stroke="#10b981" stroke-width="2" marker-end="url(#arrow-fwd)"/>
  <rect x="460" y="45" width="100" height="30" rx="4" fill="#1e293b" stroke="#10b981" stroke-width="1.5"/>
  <text x="510" y="65" fill="#10b981" font-size="11" text-anchor="middle">Output: Y</text>
  <!-- Tangent Stream (Dual Numbers) -->
  <rect x="30" y="95" width="100" height="30" rx="4" fill="#1e293b" stroke="#38bdf8" stroke-width="1.5"/>
  <text x="80" y="115" fill="#38bdf8" font-size="11" text-anchor="middle">Tangent: V</text>
  <line x1="130" y1="110" x2="220" y2="110" stroke="#38bdf8" stroke-width="2" marker-end="url(#arrow-tan)"/>
  <rect x="220" y="95" width="150" height="30" rx="4" fill="#1e293b" stroke="#38bdf8" stroke-width="1.5"/>
  <text x="295" y="115" fill="#f8fafc" font-size="11" text-anchor="middle">JVP: V · W</text>
  <line x1="370" y1="110" x2="460" y2="110" stroke="#38bdf8" stroke-width="2" marker-end="url(#arrow-tan)"/>
  <rect x="460" y="95" width="100" height="30" rx="4" fill="#1e293b" stroke="#38bdf8" stroke-width="1.5"/>
  <text x="510" y="115" fill="#38bdf8" font-size="11" text-anchor="middle">Tangent dY</text>
</svg>

### Cross-Domain Generalisation Matrix
| Scientific Domain | Manifestation of Forward AD & Functional Transforms |
| :--- | :--- |
| **Physics-Informed ML (SciML)** | Computing exact spatial differential operators (Laplacian $\nabla^2 u$, divergence $\nabla \cdot u$) without numerical grid artifacts via `jvp`. |
| **Robotics & Control (MPC)** | Differentiating physical robot forward kinematics ($f: \theta \to \mathbf{x}$) where input degrees of freedom are small ($D_{\text{in}} \le 7$) and output trajectories are high-dimensional. |
| **Privacy & Security (DP-SGD)** | Computing per-sample gradient vectors $\nabla_\theta \mathcal{L}_i$ via `vmap` to perform bounded clipping and calibrated Gaussian noise injection without Python loops. |

### Key Takeaways
1. **Forward AD Superiority on Tall Functions:** For functions mapping $\mathbb{R} \to \mathbb{R}^M$, forward AD evaluates the exact derivative in 1 pass compared to $M$ passes in standard backprop.
2. **Vectorization without Loops:** `torch.func.vmap` transforms single-sample operations into batched hardware tensor operations automatically.
3. **Directional Curvature in $\mathcal{O}(P)$:** Hessian-vector products $H v$ via mixed forward-over-reverse AD bypass the quadratic memory barrier of full Hessian matrices.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [04_forward_ad_and_functional_transforms_torch_func.md](../../markdown/00_foundations/04_forward_ad_and_functional_transforms_torch_func.md)
- **PyTorch 2.x Production Module:** [04_forward_ad_and_functional_transforms_torch_func.py](../../pytorch2/00_foundations/04_forward_ad_and_functional_transforms_torch_func.py)
- **Functional JAX Track:** [04_forward_ad_and_functional_transforms_torch_func.py](../../jax/00_foundations/04_forward_ad_and_functional_transforms_torch_func.py)
- **Native Rust Candle Track:** [04_forward_ad_and_functional_transforms_torch_func.rs](../../candle/00_foundations/04_forward_ad_and_functional_transforms_torch_func.rs)
- **CPU Fused Kernel:** [04_forward_ad_and_functional_transforms_torch_func_cpu.py](../../kernels/00_foundations/04_forward_ad_and_functional_transforms_torch_func_cpu.py)
- **Golden Reference Checkpoint:** [04_forward_ad_and_functional_transforms_torch_func.safetensors](../../weights/00_foundations/04_forward_ad_and_functional_transforms_torch_func.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 03. Autograd Mechanics & Complex Wirtinger](03_autograd_mechanics_complex_wirtinger_and_custom_functions.ipynb) | [Master Index: README.md](../../README.md) | Next: 05. Hooks & Saved Tensor Hooks (Upcoming) &rarr;
